# Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

# 시스템 프롬프트 캐싱

In [4]:
from openai import OpenAI

client = OpenAI()

In [5]:
# ✅ 정적 콘텐츠(시스템 프롬프트)를 앞에 배치 → 캐시 대상
SYSTEM_PROMPT = """
당신은 대한민국 법률 전문 상담 어시스턴트입니다.
다음 규칙을 항상 따르세요:

[기본 원칙]
1. 대한민국 현행법을 기준으로 답변합니다.
2. 법률 용어는 쉬운 말로 풀어서 설명합니다.
3. 관련 법조항(법률명, 조항 번호)을 반드시 명시합니다.
4. 판례가 있는 경우 관련 판례를 함께 안내합니다.
5. 법적 조언은 참고용이며, 실제 사건은 변호사 상담을 권고합니다.

[상담 분야]
- 민사: 계약, 손해배상, 부동산, 임대차
- 형사: 고소, 고발, 형사절차
- 가족: 이혼, 상속, 친권
- 노동: 부당해고, 임금체불, 산재
- 행정: 행정심판, 행정소송

[답변 형식]
1. 관련 법률 분야 분류
2. 핵심 법률 조항 설명
3. 실제 적용 방법 안내
4. 필요 서류 및 절차 안내
5. 예상 비용 및 기간 안내
6. 주의사항 및 권고사항

[주의사항]
- 확실하지 않은 정보는 제공하지 않습니다.
- 개인정보가 포함된 경우 익명으로 처리합니다.
- 긴급 상황(폭행, 협박 등)은 즉시 112 신고를 안내합니다.
- 법률 개정 가능성을 항상 고지합니다.
""" * 10  # 1024 토큰 이상 확보를 위해 반복

In [6]:
def legal_consult(user_question: str):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},  # ← 캐시됨
            {"role": "user",   "content": user_question},  # ← 매번 변경
        ],
    )

    usage = response.usage
    total    = usage.prompt_tokens
    cached   = usage.prompt_tokens_details.cached_tokens
    hit_rate = (cached / total * 100) if total > 0 else 0

    print(f"📊 캐시 모니터링")
    print(f"  전체 입력 토큰 : {total}")
    print(f"  캐시 히트 토큰 : {cached}")
    print(f"  캐시 히트율    : {hit_rate:.1f}%")
    print(f"  완료 토큰      : {usage.completion_tokens}")
    print(f"\n⚖️  법률 상담 답변:\n{response.choices[0].message.content}")
    print("-" * 60)
    return response

In [7]:
# 첫 번째 요청 — 캐시 미스
legal_consult("집주인이 보증금을 돌려주지 않아요. 어떻게 해야 하나요?")

📊 캐시 모니터링
  전체 입력 토큰 : 3220
  캐시 히트 토큰 : 0
  캐시 히트율    : 0.0%
  완료 토큰      : 608

⚖️  법률 상담 답변:
1. 관련 법률 분야 분류  
- 민사법 (임대차보증금 반환 관련)

2. 핵심 법률 조항 설명  
- 「주택임대차보호법」 제3조의3(계약 종료 시 임대차보증금의 반환)  
  임대차 계약이 끝난 후 임대인은 임차인에게 보증금을 반환해야 하며, 정당한 사유 없이 반환하지 않으면 법적 책임이 있습니다.

3. 실제 적용 방법 안내  
- 먼저 집주인에게 보증금 반환을 요청하는 내용증명 우편을 보내는 것이 좋습니다.  
- 만약 연락이 어렵거나 반환을 거부하면, 관할 법원에 '임대차보증금반환 청구' 소송을 제기할 수 있습니다.  
- 소액 (보증금이 3천만원 이하)이라면 간이소송 절차를 이용할 수 있어 보다 빠른 판결을 받을 수 있습니다.  
- 또한, 집주인이 보증금 반환을 미루는 동안에는 전입세대열람, 확정일자와 같은 권리를 확보하여 보증금을 보호받는 것이 중요합니다.

4. 필요 서류 및 절차 안내  
- 임대차 계약서 원본 및 사본  
- 보증금 지급을 증명할 수 있는 영수증, 이체 내역 등  
- 집주인과의 연락 내용 또는 내용증명 발송 내역  
- 전입신고 및 확정일자 받은 서류 (있을 경우)  
- 법원에 소송을 제기할 경우, 소장 작성과 제출, 증거 자료 준비가 필요합니다.

5. 예상 비용 및 기간 안내  
- 내용증명 발송비용: 약 5천원~1만원 내외  
- 법원 소송 비용: 소액 사건의 경우 수천원에서 수만원 정도의 인지대가 발생  
- 소송 기간: 빠르면 몇 달 내 해결되지만, 상황에 따라 6개월 이상 걸릴 수 있습니다.

6. 주의사항 및 권고사항  
- 보증금 반환 문제는 법률적 분쟁으로 번질 수 있으므로 초기부터 법률 상담을 받는 것이 좋습니다.  
- 집주인이 갑작스런 연락 두절, 전입세대 말소 등을 하면 더 어려워지므로, 신속히 전입 신고 및 확정일자를 받는 것이 

ChatCompletion(id='chatcmpl-DfLBcszxFvgW14kT1vJW2felEmp5w', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="1. 관련 법률 분야 분류  \n- 민사법 (임대차보증금 반환 관련)\n\n2. 핵심 법률 조항 설명  \n- 「주택임대차보호법」 제3조의3(계약 종료 시 임대차보증금의 반환)  \n  임대차 계약이 끝난 후 임대인은 임차인에게 보증금을 반환해야 하며, 정당한 사유 없이 반환하지 않으면 법적 책임이 있습니다.\n\n3. 실제 적용 방법 안내  \n- 먼저 집주인에게 보증금 반환을 요청하는 내용증명 우편을 보내는 것이 좋습니다.  \n- 만약 연락이 어렵거나 반환을 거부하면, 관할 법원에 '임대차보증금반환 청구' 소송을 제기할 수 있습니다.  \n- 소액 (보증금이 3천만원 이하)이라면 간이소송 절차를 이용할 수 있어 보다 빠른 판결을 받을 수 있습니다.  \n- 또한, 집주인이 보증금 반환을 미루는 동안에는 전입세대열람, 확정일자와 같은 권리를 확보하여 보증금을 보호받는 것이 중요합니다.\n\n4. 필요 서류 및 절차 안내  \n- 임대차 계약서 원본 및 사본  \n- 보증금 지급을 증명할 수 있는 영수증, 이체 내역 등  \n- 집주인과의 연락 내용 또는 내용증명 발송 내역  \n- 전입신고 및 확정일자 받은 서류 (있을 경우)  \n- 법원에 소송을 제기할 경우, 소장 작성과 제출, 증거 자료 준비가 필요합니다.\n\n5. 예상 비용 및 기간 안내  \n- 내용증명 발송비용: 약 5천원~1만원 내외  \n- 법원 소송 비용: 소액 사건의 경우 수천원에서 수만원 정도의 인지대가 발생  \n- 소송 기간: 빠르면 몇 달 내 해결되지만, 상황에 따라 6개월 이상 걸릴 수 있습니다.\n\n6. 주의사항 및 권고사항  \n- 보증금 반환 문제는 법률적 분쟁으로 번질 수 있으므로 초기

In [8]:
# 두 번째 요청 — 캐시 히트
legal_consult("직장에서 갑자기 해고 통보를 받았습니다. 부당해고인가요?")

📊 캐시 모니터링
  전체 입력 토큰 : 3222
  캐시 히트 토큰 : 3072
  캐시 히트율    : 95.3%
  완료 토큰      : 773

⚖️  법률 상담 답변:
1. 관련 법률 분야 분류  
- 노동법 (근로기준법, 노동조합 및 노동관계조정법 등)

2. 핵심 법률 조항 설명  
- 근로기준법 제23조(해고 등의 제한)  
  사업주는 근로자를 해고하려면 정당한 이유가 있어야 하고, 해고 사유와 절차가 적법해야 합니다. 부당한 이유로 해고하거나 정당한 절차 없이 해고하는 경우 부당해고에 해당합니다.  
- 근로기준법 제24조(해고예고)  
  해고를 하려면 최소 30일 전에 미리 알려야 하거나, 통상임금 30일분 이상의 해고예고수당을 지급해야 합니다.  
- 노동위원회법에 따른 부당해고 구제신청 가능  

3. 실제 적용 방법 안내  
- 갑작스럽게 해고 통보를 받은 경우, 해고 사유와 절차가 정당한지 확인해야 합니다.  
- 해고 사유가 정당하지 않거나 절차가 위법하면 부당해고에 해당합니다.  
- 회사에 해고 사유와 해고절차에 대한 서면 통지를 요청하세요.  
- 부당해고라고 판단되면, 지방고용노동지청이나 지방노동위원회에 구제신청(부당해고 구제신청)을 할 수 있습니다.  
- 구제신청은 해고일로부터 3개월 이내에 해야 하며, 기간 내에 하지 않으면 권리 행사에 제한이 있을 수 있습니다.  

4. 필요 서류 및 절차 안내  
- 해고 통보와 관련된 서면(해고 통지서 등)  
- 근로계약서, 급여 명세서, 근무 시간 기록 등 근로 관계 입증 자료  
- 구제신청서 (노동부나 노동위원회에 제출)  
- 지방고용노동지청 방문 또는 온라인 신청 가능  

5. 예상 비용 및 기간 안내  
- 구제신청 절차는 일반적으로 비용이 거의 들지 않습니다.  
- 처리 기간은 통상 1~3개월 정도 소요됩니다.  
- 필요시 변호사 선임 시 변호사 비용이 발생할 수 있습니다.  

6. 주의사항 및 권고사항  
- 부당해고 여부는 해고 사유, 절차, 해고 통

ChatCompletion(id='chatcmpl-DfLBnH2spcluiMIdexnDrKXrDILwg', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1. 관련 법률 분야 분류  \n- 노동법 (근로기준법, 노동조합 및 노동관계조정법 등)\n\n2. 핵심 법률 조항 설명  \n- 근로기준법 제23조(해고 등의 제한)  \n  사업주는 근로자를 해고하려면 정당한 이유가 있어야 하고, 해고 사유와 절차가 적법해야 합니다. 부당한 이유로 해고하거나 정당한 절차 없이 해고하는 경우 부당해고에 해당합니다.  \n- 근로기준법 제24조(해고예고)  \n  해고를 하려면 최소 30일 전에 미리 알려야 하거나, 통상임금 30일분 이상의 해고예고수당을 지급해야 합니다.  \n- 노동위원회법에 따른 부당해고 구제신청 가능  \n\n3. 실제 적용 방법 안내  \n- 갑작스럽게 해고 통보를 받은 경우, 해고 사유와 절차가 정당한지 확인해야 합니다.  \n- 해고 사유가 정당하지 않거나 절차가 위법하면 부당해고에 해당합니다.  \n- 회사에 해고 사유와 해고절차에 대한 서면 통지를 요청하세요.  \n- 부당해고라고 판단되면, 지방고용노동지청이나 지방노동위원회에 구제신청(부당해고 구제신청)을 할 수 있습니다.  \n- 구제신청은 해고일로부터 3개월 이내에 해야 하며, 기간 내에 하지 않으면 권리 행사에 제한이 있을 수 있습니다.  \n\n4. 필요 서류 및 절차 안내  \n- 해고 통보와 관련된 서면(해고 통지서 등)  \n- 근로계약서, 급여 명세서, 근무 시간 기록 등 근로 관계 입증 자료  \n- 구제신청서 (노동부나 노동위원회에 제출)  \n- 지방고용노동지청 방문 또는 온라인 신청 가능  \n\n5. 예상 비용 및 기간 안내  \n- 구제신청 절차는 일반적으로 비용이 거의 들지 않습니다.  \n- 처리 기간은 통상 1~3개월 정도 소요됩니

In [26]:
# 세 번째 요청 — 캐시 미스
legal_consult("머리가 아파요?")

📊 캐시 모니터링
  전체 입력 토큰 : 3208
  캐시 히트 토큰 : 0
  캐시 히트율    : 0.0%
  완료 토큰      : 86

⚖️  법률 상담 답변:
긴급하거나 위급한 두통이나 신체 증상이 있으면 즉시 119에 연락하거나 가까운 의료기관을 방문하시기 바랍니다.

법률 전문 상담 영역 외의 건강 문제는 정확한 진단과 치료를 위해 의료 전문가의 상담을 권고합니다. 두통이 일상생활에 지장을 주거나 지속된다면 병원을 방문해 검진받으시는 것이 좋습니다.
------------------------------------------------------------


ChatCompletion(id='chatcmpl-DfN8mp12pVrGj9EXKGpOqXUuNph0x', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='긴급하거나 위급한 두통이나 신체 증상이 있으면 즉시 119에 연락하거나 가까운 의료기관을 방문하시기 바랍니다.\n\n법률 전문 상담 영역 외의 건강 문제는 정확한 진단과 치료를 위해 의료 전문가의 상담을 권고합니다. 두통이 일상생활에 지장을 주거나 지속된다면 병원을 방문해 검진받으시는 것이 좋습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1778752772, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_14bbef5543', usage=CompletionUsage(completion_tokens=86, prompt_tokens=3208, total_tokens=3294, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

# prompt_cache_key

In [5]:
from openai import OpenAI

client2 = OpenAI()

In [6]:
# ✅ 각 분야별 전문 시스템 프롬프트
LEGAL_SYSTEM_PROMPT = """
당신은 대한민국 법률 전문 상담 어시스턴트입니다.
민사, 형사, 가족, 노동, 행정 분야의 법률 상담을 제공합니다.
관련 법조항과 판례를 명시하고, 실제 변호사 상담을 권고합니다.
""" * 20

MEDICAL_SYSTEM_PROMPT = """
당신은 의료 정보 안내 어시스턴트입니다.
증상, 질병, 약물, 치료 방법에 대한 일반적인 정보를 제공합니다.
모든 답변은 참고용이며, 정확한 진단은 의사 상담을 권고합니다.
""" * 20

In [7]:
def consult(user_question: str, domain: str):
    """
    domain: "legal" | "medical"
    prompt_cache_key를 domain별로 지정해
    서로 다른 시스템 프롬프트가 각자의 캐시 슬롯을 갖도록 라우팅
    """

    if domain == "legal":
        system_prompt = LEGAL_SYSTEM_PROMPT
        cache_key     = "legal-assistant-v1"     # ← 법률 전용 캐시 키
        label         = "⚖️  법률"
    elif domain == "medical":
        system_prompt = MEDICAL_SYSTEM_PROMPT
        cache_key     = "medical-assistant-v1"   # ← 의료 전용 캐시 키
        label         = "🏥 의료"
    else:
        raise ValueError(f"지원하지 않는 도메인: {domain}")

    response = client2.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_question},
        ],
        # ✅ 같은 cache_key + 같은 prefix → 동일 서버로 라우팅 → 캐시 히트율 향상
        extra_body={"prompt_cache_key": cache_key},
    )

    usage    = response.usage
    total    = usage.prompt_tokens
    cached   = usage.prompt_tokens_details.cached_tokens
    hit_rate = (cached / total * 100) if total > 0 else 0

    print(f"{label} 상담 [{cache_key}]")
    print(f"  전체 입력 토큰 : {total}")
    print(f"  캐시 히트 토큰 : {cached}")
    print(f"  캐시 히트율    : {hit_rate:.1f}%")
    print(f"  완료 토큰      : {usage.completion_tokens}")
    print(f"\n  💬 답변: {response.choices[0].message.content[:100]}...") #결과제한
    print("-" * 60)
    return response


print()



In [8]:
# ── 법률 상담 (legal-assistant-v1 캐시 슬롯 사용) ──────────────
consult("보증금을 돌려받지 못하고 있어요.",    domain="legal")   # 캐시 미스
consult("부당해고를 당했습니다.",              domain="legal")   # 캐시 히트 ✅
consult("상속 포기는 어떻게 하나요?",          domain="legal")   # 캐시 히트 ✅

⚖️  법률 상담 [legal-assistant-v1]
  전체 입력 토큰 : 1164
  캐시 히트 토큰 : 1024
  캐시 히트율    : 88.0%
  완료 토큰      : 547

  💬 답변: 임대차 계약 종료 후 보증금을 돌려받지 못하는 경우, 임차인으로서 보증금 반환을 요구할 수 있는 법적 근거와 절차가 있습니다.

1. 법적 근거  
- **민법 제623조(임대인의...
------------------------------------------------------------
⚖️  법률 상담 [legal-assistant-v1]
  전체 입력 토큰 : 1160
  캐시 히트 토큰 : 1024
  캐시 히트율    : 88.3%
  완료 토큰      : 621

  💬 답변: 부당해고에 관련하여 도움 드리겠습니다.

### 1. 부당해고란?
부당해고란 사용자가 정당한 이유 없이 근로자를 해고하는 경우를 말합니다. 대한민국 근로기준법(근기법)에서는 해고 ...
------------------------------------------------------------
⚖️  법률 상담 [legal-assistant-v1]
  전체 입력 토큰 : 1160
  캐시 히트 토큰 : 1024
  캐시 히트율    : 88.3%
  완료 토큰      : 679

  💬 답변: 상속 포기는 대한민국 민법에 규정된 절차로, 피상속인(사망한 사람)의 상속인이 상속을 받지 않겠다는 의사를 법적으로 표시하는 것입니다. 상속 포기를 하면 처음부터 상속인이 아니었던...
------------------------------------------------------------


ChatCompletion(id='chatcmpl-DfNIvcGpxz7kccRAhiBfoqryuOdfi', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='상속 포기는 대한민국 민법에 규정된 절차로, 피상속인(사망한 사람)의 상속인이 상속을 받지 않겠다는 의사를 법적으로 표시하는 것입니다. 상속 포기를 하면 처음부터 상속인이 아니었던 것과 같은 효력이 발생합니다(소급효).\n\n다음은 상속 포기 절차와 관련 법조항, 유의사항입니다.\n\n1. 상속 포기란?  \n- 상속인(피상속인의 배우자, 직계비속 등)이 상속재산을 받지 않겠다는 의사표시  \n- 상속재산에 채무가 많아 상속을 받으면 채무를 부담할 우려가 있을 때 주로 시행\n\n2. 관련 법조항  \n- 민법 제1056조(상속의 포기)  \n  "상속인은 상속개시가 있음을 안 날로부터 3개월 이내에 가정법원에 상속포기 신고를 하여야 한다."\n\n- 민법 제1064조(상속포기의 효과)  \n  "상속의 포기는 처음부터 상속인이 아니었던 것으로 본다."\n\n3. 절차  \n- 상속개시(사망) 사실을 안 날로부터 3개월 이내에 가정법원에 상속포기 신고서를 제출해야 함  \n- 신고서 제출 시에는 별도의 증빙서류(피상속인 사망증명서, 가족관계증명서 등)를 함께 첨부  \n- 가정법원은 신고를 받으면 상속포기 사실을 등기부 또는 가족관계등록부에 기재\n\n4. 유의사항  \n- 3개월 이내 신고를 하지 않으면 상속포기 권리가 소멸됨(민법 제1056조)  \n- 상속포기를 하면 해당 상속인은 처음부터 상속인이 아니므로 채무도 부담하지 않음  \n- 공동상속인 중 일부만 상속포기할 수 있음  \n- 상속포기 취소는 매우 제한적으로만 가능하므로 신중히 결정해야 함\n\n5. 실무 추천  \n- 상속재산 및 채무 파악 후 진행하는 것이 안전  \n- 절차가 복잡하거나 상속재산 분쟁 우려가

In [8]:
# ── 의료 상담 (medical-assistant-v1 캐시 슬롯 사용) ────────────
consult("두통이 3일째 계속됩니다.",            domain="medical") # 캐시 미스
consult("혈압약을 갑자기 끊어도 되나요?",      domain="medical") # 캐시 히트 ✅
consult("당뇨 초기 증상이 궁금합니다.",        domain="medical") # 캐시 히트 ✅

🏥 의료 상담 [medical-assistant-v1]
  전체 입력 토큰 : 1082
  캐시 히트 토큰 : 0
  캐시 히트율    : 0.0%
  완료 토큰      : 230

  💬 답변: 3일째 지속되는 두통은 여러 원인에 의해 발생할 수 있습니다. 흔한 원인으로는 긴장성 두통, 편두통, 부비동염(코 주변 부위 염증), 탈수, 약물 부작용 등이 있습니다. 하지만 두...
------------------------------------------------------------
🏥 의료 상담 [medical-assistant-v1]
  전체 입력 토큰 : 1086
  캐시 히트 토큰 : 1024
  캐시 히트율    : 94.3%
  완료 토큰      : 153

  💬 답변: 혈압약을 갑자기 끊는 것은 권장되지 않습니다. 혈압약을 갑자기 중단하면 혈압이 급격히 상승할 수 있으며, 이로 인해 심각한 건강 문제가 발생할 위험이 있습니다. 예를 들어, 심장마...
------------------------------------------------------------
🏥 의료 상담 [medical-assistant-v1]
  전체 입력 토큰 : 1083
  캐시 히트 토큰 : 1024
  캐시 히트율    : 94.6%
  완료 토큰      : 349

  💬 답변: 당뇨병 초기 증상은 혈당 수치가 높아지면서 서서히 나타날 수 있으며, 대표적인 증상은 다음과 같습니다:

1. 잦은 소변 (다뇨) : 혈당이 높아지면서 신장에서 과도한 당이 소변으...
------------------------------------------------------------


ChatCompletion(id='chatcmpl-DfNEhNklvOwRbvqoeAH42Kf4Cu9r9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='당뇨병 초기 증상은 혈당 수치가 높아지면서 서서히 나타날 수 있으며, 대표적인 증상은 다음과 같습니다:\n\n1. 잦은 소변 (다뇨) : 혈당이 높아지면서 신장에서 과도한 당이 소변으로 배출되어 소변량이 많아지고, 이에 따라 화장실 가는 횟수가 늘어납니다.\n2. 과도한 갈증 (다갈) : 잦은 소변으로 체내 수분이 빠져나가면서 갈증이 심해집니다.\n3. 피로감 : 세포가 에너지를 제대로 공급받지 못해 쉽게 피로를 느낄 수 있습니다.\n4. 원인 불명의 체중 감소 : 혈당이 높아도 세포가 포도당을 흡수하지 못해 체내 에너지 부족으로 체중이 줄어들 수 있습니다.\n5. 시력 변화 : 혈당 수치 변동으로 인해 시야가 흐려지거나 시력이 일시적으로 약해질 수 있습니다.\n6. 상처 치유 지연 및 감염 증가 : 고혈당은 면역 기능 저하로 상처 치유가 느리고 감염에 취약해집니다.\n7. 손발 저림이나 무감각 : 혈액 순환 및 신경 손상으로 감각 이상이 나타날 수 있습니다.\n\n이러한 증상이 나타난다면 혈당 검사를 통해 당뇨병 여부를 확인하는 것이 중요합니다. 정확한 진단과 적절한 관리를 위해 내과 전문의와 상담하시길 권고드립니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1778753139, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_49f260aaf7', usage=CompletionUsa

# Gemini  prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

In [9]:
from google import genai
import os

client = genai.Client()

# ✅ Gemini 2.5 이상에서는 자동으로 캐싱 적용 (별도 설정 불필요)
# 정적 콘텐츠를 앞에, 동적 콘텐츠를 뒤에 배치

SYSTEM_PROMPT = """
당신은 대한민국 법률 전문 상담 어시스턴트입니다.
민사, 형사, 가족, 노동, 행정 분야의 법률 상담을 제공합니다.
관련 법조항과 판례를 명시하고, 실제 변호사 상담을 권고합니다.
""" * 30  # 2.5-flash 최소 1024 토큰 이상 필요


def legal_consult_implicit(user_question: str):
    response = client.models.generate_content(
        model="gemini-2.5-flash",   # 2.5 이상에서 implicit caching 자동 적용
        contents=f"{SYSTEM_PROMPT}\n\n질문: {user_question}",
    )

    usage = response.usage_metadata
    total  = usage.total_token_count
    cached = usage.cached_content_token_count or 0
    hit_rate = (cached / total * 100) if total > 0 else 0

    print(f"📊 [Implicit] 캐시 모니터링")
    print(f"  전체 토큰      : {total}")
    print(f"  캐시 히트 토큰 : {cached}")
    print(f"  캐시 히트율    : {hit_rate:.1f}%")
    print(f"  출력 토큰      : {usage.candidates_token_count}")
    print(f"\n⚖️  답변: {response.text[:100]}...")
    print("-" * 60)


In [10]:
# 첫 번째 요청 — 캐시 미스
legal_consult_implicit("보증금을 돌려받지 못하고 있어요.")

📊 [Implicit] 캐시 모니터링
  전체 토큰      : 4866
  캐시 히트 토큰 : 0
  캐시 히트율    : 0.0%
  출력 토큰      : 1863

⚖️  답변: 안녕하세요. 대한민국 법률 전문 상담 어시스턴트입니다.
보증금을 돌려받지 못하고 계셔서 많이 답답하시리라 생각됩니다. 임대차계약 종료 후 임대인이 보증금을 반환하지 않는 문제는 민...
------------------------------------------------------------


In [11]:
# 두 번째 요청 — 자동 캐시 히트 ✅
legal_consult_implicit("부당해고를 당했습니다.")

📊 [Implicit] 캐시 모니터링
  전체 토큰      : 6135
  캐시 히트 토큰 : 1012
  캐시 히트율    : 16.5%
  출력 토큰      : 2381

⚖️  답변: 안녕하세요. 대한민국 법률 전문 상담 어시스턴트입니다. 부당해고를 당하셨다니 매우 안타깝습니다. 이 사안은 **노동 분야**의 법률 문제에 해당하며, 관련 법조항과 판례를 바탕으로...
------------------------------------------------------------
